In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pathlib import Path
import os 
import sys
from collections import Counter

from imblearn.under_sampling import RandomUnderSampler

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
BASE_DIR = Path.cwd() 
print(BASE_DIR)

c:\Users\Neezy\Videos\attacks_improved_v2\Attacks-Improved_v2\Random_forest


In [4]:
project_root = os.path.abspath('..')
if project_root not in sys.path: 
    sys.path.append(project_root)

In [5]:
#### log initialization
from utils.logs import logger_module 
logger = logger_module(logger_name=__name__) 

In [6]:
# import attacks
from attack_functionality.general_attack import GeneralAttack 
attack = GeneralAttack()

In [7]:
data = pd.read_csv(BASE_DIR.parent/'data/ds_with_sample_name.csv')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [8]:
X = data.iloc[:,:-1]
y = data.iloc[:,-1]

In [9]:
random_under_sample = RandomUnderSampler(random_state=42)

In [10]:
x_resample, y_resample = random_under_sample.fit_resample(X,y)
balanced_df = pd.DataFrame(x_resample, columns=X.columns)
balanced_df['label'] = y_resample

C:\Users\Neezy\AppData\Local\Temp\ipykernel_4572\380394481.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  balanced_df['label'] = y_resample


In [11]:
balanced_df = balanced_df.copy()

In [12]:
# is_all_zero = (balanced_df.iloc[:, 2:-1].values == 0).all(axis=1)
# counter = 0
# for x in is_all_zero: 
#     if x==True: 
#        counter +=1 
#     else: 
#         print("fine") 

# print(counter) 

In [13]:
balanced_df['label'].value_counts()

label
0    570
1    570
Name: count, dtype: int64

In [14]:
X_bal = balanced_df.iloc[:, 2:-1].values
y_bal = balanced_df.iloc[:, -1].values
original_indices = np.arange(len(X_bal)) 
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X_bal, y_bal, original_indices, test_size=0.2, random_state=42)
classifier = RandomForestClassifier(n_estimators=100, random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n {cm}") 
cr = classification_report(y_test, y_pred)
print(f"Classification report:\n {cr}")

Accuracy: 94.74%
Confusion Matrix:
 [[124   4]
 [  8  92]]
Classification report:
               precision    recall  f1-score   support

           0       0.94      0.97      0.95       128
           1       0.96      0.92      0.94       100

    accuracy                           0.95       228
   macro avg       0.95      0.94      0.95       228
weighted avg       0.95      0.95      0.95       228



In [15]:
print(X_train.shape)

(912, 277)


In [16]:
print(y_train.shape)

(912,)


In [17]:
xtrain_candidate_num = round(0.2*len(X_train))
print(xtrain_candidate_num)

182


In [18]:
ransomware_training_sample_indices = np.where(y_train==1)[0]
non_zero_mask = X_train[ransomware_training_sample_indices].sum(axis=1) !=0
valid_indices = ransomware_training_sample_indices[non_zero_mask]

idx = np.random.choice(valid_indices, 182, replace=False)
xtrain_poison_candidates = X_train[idx] 


In [19]:
print(len(xtrain_poison_candidates))

182


In [20]:
perturbation=90
asamples, adv_idx = attack.generate_adversarial_sample(idx, xtrain_poison_candidates, perturbation, classifier)

The average  l2 distance 130.31383652431037 across 162 samples
The biggest  l2 distance 167.14664220378464 across 162 samples
The smallest l2 distance 83.46256645946133 across 162 samples


In [21]:
print(len(asamples))
print(f"AGR: {len(asamples)/len(xtrain_poison_candidates):.4f}") 

162
AGR: 0.8901


In [22]:
### Adversarial Retrain ###  
mask = np.ones(X_train.shape[0], dtype=bool)
mask[adv_idx] = False

X_train_base_samples_removed = X_train[mask]
y_train_base_samples_removed = y_train[mask]
print(f"length of Xtrain before successful base adversarial samples removed: {len(X_train)}")
print(f"length of Xtrain with successful base adversarial samples removed: {len(X_train_base_samples_removed)}")

### Select a hold out test set of the generated adversarial samples
k = round(0.2 * len(asamples)) 
print(f"20% of the adversarial samples held out for testing: {k}")
remove_samples_from_asamples = random.sample(asamples, k)
print(len(remove_samples_from_asamples))

### remove test set from total adversarial samples and add adversarial samples to the training set 
result = [item for item in asamples if item not in remove_samples_from_asamples]
np_result = np.array(result)
print(len(result))
poisoned_xtrain = np.concatenate((X_train_base_samples_removed, np_result))
print(f"Total length of poisoned Xtrain: {len(poisoned_xtrain)}")

### Add the corresponding amount of ytrain adversarial samples. (correctly labelled as ransomware) 
adv_y = []
for i in range(len(np_result)): 
    adv_y.append(1) 
    
print(len(adv_y))

np_advy = np.array(adv_y)
poisoned_ytrain = np.concatenate((y_train_base_samples_removed, np_advy))
print(f"Total length of poisoned Ytrain: {len(poisoned_ytrain)}") 

if len(poisoned_ytrain) == len(poisoned_xtrain): 
    print("Poisoned Xtrain and ytrain are aligned, posioned training can now commence")
else: 
    print("Not aligned, something is not aligned, check k value")

length of Xtrain before successful base adversarial samples removed: 912
length of Xtrain with successful base adversarial samples removed: 750
20% of the adversarial samples held out for testing: 32
32
118
Total length of poisoned Xtrain: 868
118
Total length of poisoned Ytrain: 868
Poisoned Xtrain and ytrain are aligned, posioned training can now commence


In [23]:
poisoned_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
poisoned_classifier.fit(poisoned_xtrain, poisoned_ytrain)
y_pred_poisoned = poisoned_classifier.predict(X_test)
poisoned_accuracy = accuracy_score(y_test, y_pred_poisoned)
print(f'Accuracy trained on poisoned samples: {poisoned_accuracy * 100:.2f}%')
cm_poisoned = confusion_matrix(y_test, y_pred_poisoned)
print(f"Confusion Matrix trained on poisoned samples:\n {cm_poisoned}") 
cr_poisoned = classification_report(y_test, y_pred_poisoned)
print(f"Classification report trained on poisoned samples:\n {cr_poisoned}")

Accuracy trained on poisoned samples: 95.18%
Confusion Matrix trained on poisoned samples:
 [[125   3]
 [  8  92]]
Classification report trained on poisoned samples:
               precision    recall  f1-score   support

           0       0.94      0.98      0.96       128
           1       0.97      0.92      0.94       100

    accuracy                           0.95       228
   macro avg       0.95      0.95      0.95       228
weighted avg       0.95      0.95      0.95       228



In [24]:
def make_pred_poisoned(samples):
    predictions = [] 
    for sample in samples:
        single_pred = poisoned_classifier.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned(remove_samples_from_asamples)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc

In [25]:
###Model transfer ###
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [26]:
### Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Random Forest ####
classifier_dt = DecisionTreeClassifier(random_state=42)
classifier_dt.fit(X_train, y_train)
y_pred_rf = classifier_dt.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf) 
print(f"Accuracy of Decision Tree Target Model: {accuracy_score(y_test, y_pred):.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model = XGBClassifier(**params)
model.fit(X_train, y_train)
y_pred_xgb = model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f'Accuracy of the XGboost Model: {accuracy_xgb * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier = SVC(kernel='poly', C=4.0, random_state=42)
svm_classifier.fit(X_train, y_train)
y_pred = svm_classifier.predict(X_test)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test, y_pred):.2f}")

### SVM Classifier Kernel=Linear ###
svm_classifier_linear = SVC(kernel='linear', C=4.0, random_state=42)
svm_classifier_linear.fit(X_train, y_train)
y_pred = svm_classifier_linear.predict(X_test)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test, y_pred):.2f}")

### Logistic Regression ###
LR_model = LogisticRegression(solver='saga',max_iter=5000)
LR_model.fit(X_train, y_train)
ypred_LR = LR_model.predict(X_test)
accuracy = accuracy_score(y_test, ypred_LR)
print(f"accuracy of the Logistic Regression Model: {round(accuracy,2)}")

Accuracy of Decision Tree Target Model: 0.95
Accuracy of the XGboost Model: 91.67%
Accuracy of the SVM Classifier (Kernel=Poly): 0.46
Accuracy of the SVM Classifier (Kernel=Linear) : 0.95
accuracy of the Logistic Regression Model: 0.85


In [27]:
def evasion_attack(adversarial_samples,
                   DT_model,
                   XGB_Model, 
                   SVM_poly_model, 
                   SVM_linear_model, 
                   Logistic_reg_model):
    
    predictions = {
        "Decision_Tree":[],
        "XGB":[],
        "SVM_poly": [], 
        "SVM_Linear":[], 
        "Logistic_Regression":[]   
    }
    
    models = {
        "Decision_Tree": DT_model,
        "XGB":XGB_Model,
        "SVM_poly": SVM_poly_model, 
        "SVM_Linear":SVM_linear_model, 
        "Logistic_Regression":Logistic_reg_model
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [28]:
res = evasion_attack(asamples, classifier_dt, model, svm_classifier, svm_classifier_linear, LR_model)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = balanced_df.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names))      


 Decision_Tree: predictions

 Decision_Tree evasion rate: 96/162 samples evaded

 evaded samples:  Counter({'GandCrab': 11, 'None': 11, 'RedLineStealer': 5, 'NominatusStorm': 5, 'Babuk': 5, 'Dharma': 4, 'Azov': 4, 'Chaos': 4, 'Conti': 3, 'Sodinokibi': 3, 'DarkSide': 3, 'Nitro': 2, 'BlueSky': 2, 'Egregor': 2, 'Hive': 2, 'Erica': 1, 'Makop': 1, 'AXLocker': 1, 'RagnarLocker': 1, 'Samsam': 1, 'Ransomare.NominatusStrike': 1, 'Stop': 1, 'WannaCry': 1, 'HavanaCrypt': 1, 'DoejoCrypt': 1, 'DeriaLock': 1, 'Exorcist': 1, 'Snatch': 1, 'Nefilim': 1, 'Thanos': 1, 'SamSam': 1, 'LegionLocker': 1, 'Sekhmet': 1, 'Win32.SdBot': 1, 'XFilesStealer': 1, 'Wannacry': 1, 'InfinityLock': 1, 'Release': 1, 'Maze': 1, 'Adylkuzz': 1, 'LimeRAT': 1, '7ev3n': 1, 'Generic Trojan': 1, 'Clop': 1, 'QNAPCrypt': 1})

 XGB: predictions

 No samples evaded for XGB

 XGB evasion rate: 0/162 samples evaded

 evaded samples:  Counter()

 SVM_poly: predictions

 No samples evaded for SVM_poly

 SVM_poly evasion rate: 0/162 sampl